# Notebook 05 — Voltage Sweeps and Trap Optimization Maps

This notebook extends the reduced ion-trap workflow by sweeping electrode voltages and tracking how the pseudopotential landscape changes.

It focuses on:

- solving Laplace's equation across a 2D voltage sweep
- computing electric fields and normalized pseudopotentials
- locating a candidate interior trap point for each voltage pair
- extracting local curvature information from the Hessian
- mapping trap height and curvature-based strength across parameter space

This provides a first optimization-style workflow for comparing configurations.


## Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


## Grid and geometry

We keep the same simple surface-style geometry as in the previous notebooks, but now sweep the applied voltages.


In [ ]:
# Domain size and resolution
nx, ny = 121, 81
x = np.linspace(-3.0, 3.0, nx)
y = np.linspace(0.0, 4.0, ny)
dx = x[1] - x[0]
dy = y[1] - y[0]

# Electrode masks on the lower boundary
left_dc = (x >= -2.2) & (x <= -0.9)
center_rf = (x >= -0.5) & (x <= 0.5)
right_dc = (x >= 0.9) & (x <= 2.2)
remaining_ground = ~(left_dc | center_rf | right_dc)

print(f'Grid: {nx} x {ny}')
print(f'dx = {dx:.3f}, dy = {dy:.3f}')


## Helper functions

We define helper functions to:

- build the electrostatic boundary conditions
- solve Laplace's equation
- compute the electric field and normalized pseudopotential
- locate a candidate interior trap point
- compute the local Hessian and curvature-based trap metrics


In [ ]:
def idx(i, j, nx):
    return i * nx + j

def build_boundary_problem(v_rf, v_dc):
    V = np.zeros((ny, nx), dtype=float)
    fixed = np.zeros((ny, nx), dtype=bool)

    # Ground outer boundaries
    V[:, 0] = 0.0
    V[:, -1] = 0.0
    V[-1, :] = 0.0
    fixed[:, 0] = True
    fixed[:, -1] = True
    fixed[-1, :] = True

    # Lower boundary electrode pattern
    bottom = 0
    V[bottom, left_dc] = v_dc
    V[bottom, center_rf] = v_rf
    V[bottom, right_dc] = v_dc
    V[bottom, remaining_ground] = 0.0
    fixed[bottom, :] = True
    return V, fixed

def solve_laplace(V, fixed):
    N = nx * ny
    A = lil_matrix((N, N), dtype=float)
    b = np.zeros(N, dtype=float)

    cx = 1.0 / dx**2
    cy = 1.0 / dy**2
    c0 = -2.0 * (cx + cy)

    for i in range(ny):
        for j in range(nx):
            k = idx(i, j, nx)

            if fixed[i, j]:
                A[k, k] = 1.0
                b[k] = V[i, j]
            else:
                A[k, idx(i, j, nx)] = c0
                A[k, idx(i, j - 1, nx)] = cx
                A[k, idx(i, j + 1, nx)] = cx
                A[k, idx(i - 1, j, nx)] = cy
                A[k, idx(i + 1, j, nx)] = cy

    sol = spsolve(A.tocsr(), b)
    return sol.reshape((ny, nx))

def compute_pseudopotential(Vsol):
    dV_dy, dV_dx = np.gradient(Vsol, dy, dx)
    Ex = -dV_dx
    Ey = -dV_dy
    E_mag = np.sqrt(Ex**2 + Ey**2)
    Phi_pseudo = E_mag**2
    return Ex, Ey, E_mag, Phi_pseudo

def find_candidate_point(Phi_pseudo):
    margin_i_low = 2
    margin_i_high = 3
    margin_j = 3

    i_slice = slice(margin_i_low, ny - margin_i_high)
    j_slice = slice(margin_j, nx - margin_j)
    Phi_search = Phi_pseudo[i_slice, j_slice]

    min_flat = np.argmin(Phi_search)
    min_i_local, min_j_local = np.unravel_index(min_flat, Phi_search.shape)
    min_i = min_i_local + margin_i_low
    min_j = min_j_local + margin_j
    return min_i, min_j

def second_derivatives(F, i, j):
    d2F_dx2 = (F[i, j + 1] - 2.0 * F[i, j] + F[i, j - 1]) / dx**2
    d2F_dy2 = (F[i + 1, j] - 2.0 * F[i, j] + F[i - 1, j]) / dy**2
    d2F_dxdy = (
        F[i + 1, j + 1] - F[i + 1, j - 1] - F[i - 1, j + 1] + F[i - 1, j - 1]
    ) / (4.0 * dx * dy)
    return d2F_dx2, d2F_dy2, d2F_dxdy

def trap_metrics_from_pseudopotential(Phi_pseudo):
    i, j = find_candidate_point(Phi_pseudo)
    trap_x = x[j]
    trap_y = y[i]
    trap_value = Phi_pseudo[i, j]

    center = Phi_pseudo[i, j]
    neighbors = np.array([
        Phi_pseudo[i - 1, j],
        Phi_pseudo[i + 1, j],
        Phi_pseudo[i, j - 1],
        Phi_pseudo[i, j + 1],
    ])
    is_local_min = np.all(center <= neighbors)

    d2x, d2y, d2xy = second_derivatives(Phi_pseudo, i, j)
    H = np.array([[d2x, d2xy], [d2xy, d2y]], dtype=float)
    eigvals, eigvecs = np.linalg.eigh(H)
    omega_proxy = np.sqrt(np.clip(eigvals, 0.0, None))

    trap_strength = np.sum(np.clip(eigvals, 0.0, None))
    min_curvature = np.min(eigvals)

    return {
        'i': i,
        'j': j,
        'trap_x': trap_x,
        'trap_y': trap_y,
        'trap_value': trap_value,
        'is_local_min': bool(is_local_min),
        'H': H,
        'eigvals': eigvals,
        'omega_proxy': omega_proxy,
        'trap_strength': trap_strength,
        'min_curvature': min_curvature,
    }


## Voltage sweep setup

We sweep:

- `v_rf`: center RF-like electrode voltage
- `v_dc`: side DC-like electrode voltage

and evaluate trap metrics for each pair.


In [ ]:
v_rf_values = np.linspace(0.5, 2.0, 7)
v_dc_values = np.linspace(-2.0, -0.5, 7)

trap_height_map = np.zeros((len(v_dc_values), len(v_rf_values)))
trap_strength_map = np.zeros((len(v_dc_values), len(v_rf_values)))
min_curvature_map = np.zeros((len(v_dc_values), len(v_rf_values)))
is_local_min_map = np.zeros((len(v_dc_values), len(v_rf_values)), dtype=bool)

results = {}

print(f'Sweeping {len(v_dc_values) * len(v_rf_values)} voltage pairs...')


## Run the voltage sweep


In [ ]:
for idc, v_dc in enumerate(v_dc_values):
    for irf, v_rf in enumerate(v_rf_values):
        V, fixed = build_boundary_problem(v_rf=v_rf, v_dc=v_dc)
        Vsol = solve_laplace(V, fixed)
        Ex, Ey, E_mag, Phi_pseudo = compute_pseudopotential(Vsol)
        metrics = trap_metrics_from_pseudopotential(Phi_pseudo)

        trap_height_map[idc, irf] = metrics['trap_y']
        trap_strength_map[idc, irf] = metrics['trap_strength']
        min_curvature_map[idc, irf] = metrics['min_curvature']
        is_local_min_map[idc, irf] = metrics['is_local_min']

        results[(float(v_dc), float(v_rf))] = {
            'Vsol': Vsol,
            'Phi_pseudo': Phi_pseudo,
            **metrics,
        }

print('Voltage sweep complete.')


## Trap-height map

This map shows the candidate trap height as a function of the voltage pair.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    trap_height_map,
    origin='lower',
    aspect='auto',
    extent=[v_rf_values[0], v_rf_values[-1], v_dc_values[0], v_dc_values[-1]],
)
ax.set_title('Candidate Trap Height vs Voltage Pair')
ax.set_xlabel('Center RF-like voltage')
ax.set_ylabel('Side DC-like voltage')
fig.colorbar(im, ax=ax, label='Trap height')
plt.tight_layout()
plt.show()


## Trap-strength map

We use the sum of positive Hessian eigenvalues as a simple curvature-based trap-strength score.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    trap_strength_map,
    origin='lower',
    aspect='auto',
    extent=[v_rf_values[0], v_rf_values[-1], v_dc_values[0], v_dc_values[-1]],
)
ax.set_title('Curvature-Based Trap Strength vs Voltage Pair')
ax.set_xlabel('Center RF-like voltage')
ax.set_ylabel('Side DC-like voltage')
fig.colorbar(im, ax=ax, label='Trap strength')
plt.tight_layout()
plt.show()


## Minimum-curvature map

A strictly positive minimum curvature is a useful sign of a locally confining point in both principal directions.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    min_curvature_map,
    origin='lower',
    aspect='auto',
    extent=[v_rf_values[0], v_rf_values[-1], v_dc_values[0], v_dc_values[-1]],
)
ax.set_title('Minimum Principal Curvature vs Voltage Pair')
ax.set_xlabel('Center RF-like voltage')
ax.set_ylabel('Side DC-like voltage')
fig.colorbar(im, ax=ax, label='Minimum curvature')
plt.tight_layout()
plt.show()


## Local-minimum mask

This diagnostic shows where the candidate point also passes the immediate-neighbor local-minimum check.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    is_local_min_map.astype(float),
    origin='lower',
    aspect='auto',
    extent=[v_rf_values[0], v_rf_values[-1], v_dc_values[0], v_dc_values[-1]],
    vmin=0,
    vmax=1,
)
ax.set_title('Immediate-Neighbor Local-Minimum Check')
ax.set_xlabel('Center RF-like voltage')
ax.set_ylabel('Side DC-like voltage')
fig.colorbar(im, ax=ax, label='1 = local minimum')
plt.tight_layout()
plt.show()


## Example best configuration by trap strength

We inspect one configuration chosen by the maximum curvature-based trap-strength score.


In [ ]:
best_idx = np.unravel_index(np.argmax(trap_strength_map), trap_strength_map.shape)
best_idc, best_irf = best_idx
best_v_dc = float(v_dc_values[best_idc])
best_v_rf = float(v_rf_values[best_irf])
best = results[(best_v_dc, best_v_rf)]

print(f'Best configuration by trap strength: v_dc = {best_v_dc:.3f}, v_rf = {best_v_rf:.3f}')
print(f"Candidate point: x = {best['trap_x']:.3f}, y = {best['trap_y']:.3f}")
print(f"Trap strength: {best['trap_strength']:.3e}")
print(f"Minimum curvature: {best['min_curvature']:.3e}")
print(f"Local minimum check: {best['is_local_min']}")
print('Curvature eigenvalues:')
print(best['eigvals'])
print('Secular-frequency proxies:')
print(best['omega_proxy'])


In [ ]:
X, Y = np.meshgrid(x, y)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(X, Y, best['Phi_pseudo'], shading='auto')
cs = ax.contour(X, Y, best['Phi_pseudo'], levels=15)
ax.clabel(cs, inline=True, fontsize=8)
ax.scatter([best['trap_x']], [best['trap_y']], s=80, label='Candidate trap point')
ax.plot(x[left_dc], np.full(np.sum(left_dc), y[0]), linewidth=6)
ax.plot(x[center_rf], np.full(np.sum(center_rf), y[0]), linewidth=6)
ax.plot(x[right_dc], np.full(np.sum(right_dc), y[0]), linewidth=6)
ax.set_title('Best Configuration by Curvature-Based Trap Strength')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
fig.colorbar(im, ax=ax, label='Pseudo')
plt.tight_layout()
plt.show()


## Notes and next steps

This notebook adds a basic voltage-sweep workflow to the reduced ion-trap model.

Natural next steps:

- enlarge or refine the voltage grid near promising regions
- reject non-confining or weak minima more aggressively
- add geometry sweeps over electrode width and gap spacing
- include explicit physical constants for SI-scale frequency estimates
- add DC compensation fields and axial confinement structure
- turn the sweep into a true optimization loop

That progression will move the workflow toward practical trapped-ion design exploration.
